In [ ]:
#@title Cell 1 - Notebook overview

from IPython.display import display, Markdown

display(Markdown(r"""
# Simulation 02: Missing pathogen–plasmid combinations

## Purpose

Simulation 2 starts from the finalized Simulation 1 biological model and changes only one feature:

\[
\boxed{\text{some pathogen–plasmid combinations are not observed}}
\]

The central question remains:

\[
\boxed{\text{Does the effect of plasmid }j\text{ depend on the pathogen chromosome }i?}
\]

The model is unchanged:

\[
y_{ij}
=
\alpha
+
\boldsymbol{\beta}_C^T\mathbf c_i
+
\boldsymbol{\beta}_P^T\mathbf p_j
+
\mathbf c_i^T\mathbf B\mathbf p_j
+
u_i
+
\varepsilon_{ij}.
\]

The targets remain

\[
\Delta_{ij}=y_{ij}-y_{i0}
\]

and

\[
\Delta\Delta_{ik,j}=\Delta_{ij}-\Delta_{kj}.
\]

## What is kept identical to Simulation 1

- 200 pathogen chromosomal backgrounds.
- 20 blaTEM-1 plasmids plus plasmid-free state \(P_0\).
- 30 gene-centred chromosomal units.
- The same chromosomal state probabilities and biological effect sizes.
- The same empirical promoter-genotype/copy-number sampling.
- The same plasmid feature vector.
- The same sparse chromosome–plasmid interaction structure.
- 500 background SNPs for \(K\).
- \(\alpha=-2.5\).
- \(\sigma_g=0.40\), \(\sigma_e=0.32\).
- 100 simulation replicates.
- 200 representative bootstrap replicates.

## Simulation 2 change

The complete 200 × 21 dataset is generated first exactly as in Simulation 1.

Then **30% of the \(P_1,\ldots,P_{20}\) pathogen–plasmid observations are removed at random**.

All 200 plasmid-free \(P_0\) observations are retained.

Thus the model is fitted only to the incomplete observed dataset, but recovery is evaluated against the known truth for the full 200 × 20 plasmid-effect grid.

The primary Simulation 2 quantities are therefore the recovery of:

1. \(\Delta_{ij}\) for pathogen–plasmid combinations that were deliberately hidden.
2. \(\Delta\Delta_{ik,j}\) when at least one of the two pathogen–plasmid combinations was hidden.

The missing fraction is defined by one editable parameter in Cell 2.
"""))

print("Transition: Cell 2 loads the public plasmid-feature input, and defines the Simulation 2 settings.")


In [ ]:
#@title Cell 2 - Load public plasmid-feature input and define fixed settings

from pathlib import Path
import json
import time
import numpy as np
import pandas as pd

from scipy.linalg import cho_factor, cho_solve
from scipy.optimize import minimize_scalar

REPO_ROOT = Path.cwd()
DATA_FILE = REPO_ROOT / "data" / "plasmid_feature_pairs.csv"

OUTPUT_DIR = (
    REPO_ROOT
    / "results"
    / "simulation_02_missing_combinations"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_FILE.exists():
    raise FileNotFoundError(
        "Public plasmid-feature input was not found:\n"
        f"{DATA_FILE}\n"
        "Run the notebook from the repository root."
    )

empirical_pairs = pd.read_csv(
    DATA_FILE,
    low_memory=False,
)

KEY_SITE_COLUMNS = [
    "sutcliffe_32_nt",
    "sutcliffe_162_nt",
    "sutcliffe_175_nt",
]

required_columns = [
    *KEY_SITE_COLUMNS,
    "CN_TEM1",
]

missing_columns = [
    c for c in required_columns
    if c not in empirical_pairs.columns
]

if missing_columns:
    raise RuntimeError(
        "The public plasmid-feature input is missing required columns:\n"
        + "\n".join(f"  - {c}" for c in missing_columns)
    )

empirical_pairs = empirical_pairs[
    required_columns
].copy()

for col in KEY_SITE_COLUMNS:
    empirical_pairs[col] = (
        empirical_pairs[col]
        .astype(str)
        .str.strip()
        .str.upper()
    )

empirical_pairs["CN_TEM1"] = pd.to_numeric(
    empirical_pairs["CN_TEM1"],
    errors="coerce",
)

valid_nt = {"A", "C", "G", "T"}

complete_mask = (
    empirical_pairs[KEY_SITE_COLUMNS]
    .apply(lambda s: s.isin(valid_nt))
    .all(axis=1)
    & empirical_pairs["CN_TEM1"].notna()
    & (empirical_pairs["CN_TEM1"] > 0)
)

empirical_pairs = (
    empirical_pairs.loc[complete_mask]
    .reset_index(drop=True)
)

if len(empirical_pairs) < 20:
    raise RuntimeError(
        f"Only {len(empirical_pairs)} complete promoter/copy-number pairs remain; "
        "at least 20 are required."
    )

CN_REFERENCE = float(
    empirical_pairs["CN_TEM1"].median()
)

if not np.isfinite(CN_REFERENCE) or CN_REFERENCE <= 0:
    raise RuntimeError(
        "The median CN_TEM1 is not positive."
    )

# -------------------------------------------------------------------------
# Simulation 1 biological settings: unchanged
# -------------------------------------------------------------------------

MASTER_SEED = 20260906

N_PATHOGENS = 200
N_PLASMIDS = 20

PREDEFINED_GENES = [
    "acrB", "acrR", "ampC", "basR", "cirA", "cyaA", "fabI", "folP", "ftsI",
    "gyrA", "marR", "nfsA", "nfsB", "ompC", "ompF", "parC", "parE", "pmrB",
    "ptsI", "rpoB", "rpsL", "soxR", "soxS", "uhpT", "acrA", "tolC", "marA",
    "rob", "ompR", "envZ",
]

TARGET_FEATURE_LABELS = []
for gene in PREDEFINED_GENES:
    TARGET_FEATURE_LABELS.extend([
        f"{gene}:coding",
        f"{gene}:upstream_300bp",
    ])

D_C = len(TARGET_FEATURE_LABELS)

PLASMID_FEATURE_LABELS = [
    "TEM1_plasmid_presence",
    "C32T",
    "G162T",
    "G175A",
    "log2_CN_relative_to_empirical_median",
]
D_P = len(PLASMID_FEATURE_LABELS)

CHROMOSOMAL_STATES = np.array([-1.0, 0.0, 1.0])
CHROMOSOMAL_STATE_PROBS = np.array([0.15, 0.70, 0.15])

N_BACKGROUND_SNPS = 500
ALLELE_FREQ_LOW = 0.10
ALLELE_FREQ_HIGH = 0.90

ALPHA_TRUE = -2.5

SIGMA_G_TRUE = 0.40
SIGMA_E_TRUE = 0.32
SIGMA_G2_TRUE = SIGMA_G_TRUE ** 2
SIGMA_E2_TRUE = SIGMA_E_TRUE ** 2

BETA_P_TRUE = np.array([
    0.50,
    0.50,
    0.25,
    0.00,
    0.25,
], dtype=float)

INTERACTION_MAGNITUDE = 0.10

# -------------------------------------------------------------------------
# Simulation 2 change
# -------------------------------------------------------------------------

MISSING_PPLUS_FRACTION = 0.30

# Keep all P0 observations. Remove only P1...P20 combinations.
# The mask is redrawn until the observed fixed-effect matrix remains full rank.

N_SIM_REPLICATES = 100
N_BOOTSTRAP = 200

RUN_FULL_BOOTSTRAP_COVERAGE = False

print("=" * 90)
print("SIMULATION 02 — FIXED SETTINGS")
print("=" * 90)
print(f"Empirical promoter/CN pairs available: {len(empirical_pairs):,}")
print(f"Empirical median CN_TEM1:               {CN_REFERENCE:.6f}")
print(f"Pathogens:                              {N_PATHOGENS}")
print(f"Plasmids:                               {N_PLASMIDS}")
print(f"Complete observations before masking:   {N_PATHOGENS * (N_PLASMIDS + 1):,}")
print(f"Missing P+ fraction:                    {MISSING_PPLUS_FRACTION:.0%}")
print("P0 observations retained:               100%")
print(f"alpha:                                  {ALPHA_TRUE}")
print(f"sigma_g:                                {SIGMA_G_TRUE}")
print(f"sigma_e:                                {SIGMA_E_TRUE}")
print(f"Output directory:                       {OUTPUT_DIR}")
print("\nCell 2: PASS")


In [ ]:
#@title Cell 3 - Define the Simulation 1 biological model and Simulation 2 missingness

def feature_index(gene, region):
    label = (
        f"{gene}:coding"
        if region == "coding"
        else f"{gene}:upstream_300bp"
    )
    return TARGET_FEATURE_LABELS.index(label)


def build_true_chromosomal_coefficients():
    beta_C = np.zeros(D_C, dtype=float)
    B = np.zeros((D_C, D_P), dtype=float)

    efflux_machinery = {"acrA", "acrB", "tolC"}
    repressors = {"acrR", "marR"}
    activators = {"marA", "rob", "soxR", "soxS"}
    porins = {"ompC", "ompF"}

    for gene in PREDEFINED_GENES:
        coding_i = feature_index(gene, "coding")
        upstream_i = feature_index(gene, "upstream")

        if gene in efflux_machinery:
            beta_C[coding_i] = +0.25
            beta_C[upstream_i] = +0.25

        elif gene in repressors:
            beta_C[coding_i] = -0.25
            beta_C[upstream_i] = -0.25

        elif gene in activators:
            beta_C[coding_i] = +0.25
            beta_C[upstream_i] = +0.25

        elif gene in porins:
            beta_C[coding_i] = -0.25
            beta_C[upstream_i] = -0.25

        elif gene == "ampC":
            beta_C[coding_i] = +0.10
            beta_C[upstream_i] = +0.25

        elif gene == "ftsI":
            beta_C[coding_i] = -0.25
            beta_C[upstream_i] = -0.10

        if gene in efflux_machinery:
            B[coding_i, 0] = +INTERACTION_MAGNITUDE
            B[upstream_i, 0] = +INTERACTION_MAGNITUDE

        elif gene in repressors:
            B[coding_i, 0] = -INTERACTION_MAGNITUDE
            B[upstream_i, 0] = -INTERACTION_MAGNITUDE

        elif gene in activators:
            B[coding_i, 0] = +INTERACTION_MAGNITUDE
            B[upstream_i, 0] = +INTERACTION_MAGNITUDE

        elif gene in porins:
            B[coding_i, 0] = -INTERACTION_MAGNITUDE
            B[upstream_i, 0] = -INTERACTION_MAGNITUDE

    return beta_C, B


BETA_C_TRUE, B_TRUE = build_true_chromosomal_coefficients()


def simulate_targeted_chromosomal_features(rng):
    for _ in range(100):
        C = rng.choice(
            CHROMOSOMAL_STATES,
            size=(N_PATHOGENS, D_C),
            p=CHROMOSOMAL_STATE_PROBS,
        ).astype(float)

        augmented = np.column_stack([
            np.ones(N_PATHOGENS, dtype=float),
            C,
        ])

        if np.linalg.matrix_rank(augmented) == D_C + 1:
            return C

    raise RuntimeError(
        "Could not generate a full-rank pathogen chromosome matrix."
    )


def sample_empirical_plasmids(rng):
    n_empirical = len(empirical_pairs)

    for _ in range(5000):
        selected_positions = rng.choice(
            n_empirical,
            size=N_PLASMIDS,
            replace=False,
        )

        sampled = (
            empirical_pairs.iloc[selected_positions]
            .copy()
            .reset_index(drop=True)
        )

        sampled.insert(
            0,
            "plasmid_id",
            [f"P{j}" for j in range(1, N_PLASMIDS + 1)],
        )

        sampled["I_C32T"] = (
            sampled["sutcliffe_32_nt"].eq("T")
        ).astype(float)

        sampled["I_G162T"] = (
            sampled["sutcliffe_162_nt"].eq("T")
        ).astype(float)

        sampled["I_G175A"] = (
            sampled["sutcliffe_175_nt"].eq("A")
        ).astype(float)

        sampled["q_CN"] = np.log2(
            sampled["CN_TEM1"].astype(float)
            / CN_REFERENCE
        )

        P = np.column_stack([
            np.ones(N_PLASMIDS, dtype=float),
            sampled["I_C32T"].to_numpy(dtype=float),
            sampled["I_G162T"].to_numpy(dtype=float),
            sampled["I_G175A"].to_numpy(dtype=float),
            sampled["q_CN"].to_numpy(dtype=float),
        ])

        P_all = np.vstack([
            np.zeros((1, D_P), dtype=float),
            P,
        ])

        augmented_state_matrix = np.column_stack([
            np.ones(N_PLASMIDS + 1, dtype=float),
            P_all,
        ])

        if np.linalg.matrix_rank(augmented_state_matrix) == D_P + 1:
            return P, sampled

    raise RuntimeError(
        "Could not sample 20 full-rank empirical plasmid profiles."
    )


def simulate_background_relatedness(rng):
    source_frequencies = rng.uniform(
        ALLELE_FREQ_LOW,
        ALLELE_FREQ_HIGH,
        size=N_BACKGROUND_SNPS,
    )

    G = rng.binomial(
        1,
        source_frequencies,
        size=(N_PATHOGENS, N_BACKGROUND_SNPS),
    ).astype(float)

    p = G.mean(axis=0)
    Z = G - p[None, :]

    denominator = float(
        np.sum(p * (1.0 - p))
    )

    if denominator <= 0:
        raise ValueError(
            "Background-SNP relatedness denominator is not positive."
        )

    K = (Z @ Z.T) / denominator
    K = (K + K.T) / 2.0

    eigenvalues, eigenvectors = np.linalg.eigh(K)

    if eigenvalues.min() < -1e-8:
        raise ValueError(
            f"Constructed K is unexpectedly non-PSD: "
            f"minimum eigenvalue={eigenvalues.min():.6g}"
        )

    eigenvalues = np.clip(
        eigenvalues,
        0.0,
        None,
    )

    K = (
        eigenvectors * eigenvalues
    ) @ eigenvectors.T

    K = (K + K.T) / 2.0

    return K, G, p


def build_complete_design(C, P):
    P_all = np.vstack([
        np.zeros((1, D_P), dtype=float),
        P,
    ])

    n_states = N_PLASMIDS + 1

    pathogen_index = np.repeat(
        np.arange(N_PATHOGENS, dtype=int),
        n_states,
    )

    plasmid_state_index = np.tile(
        np.arange(n_states, dtype=int),
        N_PATHOGENS,
    )

    C_obs = C[pathogen_index, :]
    P_obs = P_all[plasmid_state_index, :]

    interaction = np.einsum(
        "ni,nj->nij",
        C_obs,
        P_obs,
    ).reshape(
        len(pathogen_index),
        D_C * D_P,
    )

    X = np.column_stack([
        np.ones(len(pathogen_index), dtype=float),
        C_obs,
        P_obs,
        interaction,
    ])

    return (
        X,
        pathogen_index,
        plasmid_state_index,
        P_all,
    )


def draw_correlated_host_effect(rng, K, sigma_g2):
    eigenvalues, eigenvectors = np.linalg.eigh(
        (K + K.T) / 2.0
    )

    eigenvalues = np.clip(
        eigenvalues,
        0.0,
        None,
    )

    z = rng.normal(
        0.0,
        1.0,
        size=N_PATHOGENS,
    )

    u = eigenvectors @ (
        np.sqrt(
            sigma_g2 * eigenvalues
        ) * z
    )

    return u


def simulate_complete_dataset(seed):
    """
    Generate exactly the same complete biological dataset structure as Simulation 1.
    Missingness is applied only afterwards.
    """
    rng = np.random.default_rng(seed)

    C = simulate_targeted_chromosomal_features(rng)
    P, sampled_plasmids = sample_empirical_plasmids(rng)

    (
        X,
        pathogen_index,
        plasmid_state_index,
        P_all,
    ) = build_complete_design(C, P)

    K, G_background, background_frequencies = (
        simulate_background_relatedness(rng)
    )

    theta_true = np.concatenate([
        np.array([ALPHA_TRUE], dtype=float),
        BETA_C_TRUE,
        BETA_P_TRUE,
        B_TRUE.reshape(-1),
    ])

    structural_mean = X @ theta_true

    u = draw_correlated_host_effect(
        rng,
        K,
        SIGMA_G2_TRUE,
    )

    epsilon = rng.normal(
        0.0,
        SIGMA_E_TRUE,
        size=X.shape[0],
    )

    y = (
        structural_mean
        + u[pathogen_index]
        + epsilon
    )

    return {
        "seed": int(seed),
        "C": C,
        "P": P,
        "P_all": P_all,
        "sampled_plasmids": sampled_plasmids,
        "K": K,
        "G_background": G_background,
        "background_frequencies": background_frequencies,
        "beta_C_true": BETA_C_TRUE.copy(),
        "beta_P_true": BETA_P_TRUE.copy(),
        "B_true": B_TRUE.copy(),
        "theta_true": theta_true,
        "u_true": u,
        "epsilon_true": epsilon,
        "structural_mean": structural_mean,
        "X_complete": X,
        "pathogen_index_complete": pathogen_index,
        "plasmid_state_index_complete": plasmid_state_index,
        "y_complete": y,
    }


def apply_missing_pathogen_plasmid_combinations(dataset, mask_seed):
    """
    Remove a fixed fraction of P1...P20 observations at random.

    Safeguards:
    - every P0 observation remains observed;
    - every pathogen retains at least one P+ observation;
    - every plasmid is observed in at least one pathogen;
    - the observed fixed-effect matrix remains full rank.
    """
    X_complete = dataset["X_complete"]
    state_index = dataset["plasmid_state_index_complete"]

    n_rows = X_complete.shape[0]
    p_fixed = X_complete.shape[1]

    rng = np.random.default_rng(mask_seed)

    p0_rows = np.where(state_index == 0)[0]
    pplus_rows = np.where(state_index > 0)[0]

    n_remove = int(
        round(
            MISSING_PPLUS_FRACTION
            * len(pplus_rows)
        )
    )

    for _ in range(500):
        missing_pplus_rows = rng.choice(
            pplus_rows,
            size=n_remove,
            replace=False,
        )

        observed_mask = np.ones(
            n_rows,
            dtype=bool,
        )

        observed_mask[
            missing_pplus_rows
        ] = False

        if not np.all(
            observed_mask[p0_rows]
        ):
            continue

        observed_grid = (
            observed_mask.reshape(
                N_PATHOGENS,
                N_PLASMIDS + 1,
            )[:, 1:]
        )

        if np.any(
            observed_grid.sum(axis=1) == 0
        ):
            continue

        if np.any(
            observed_grid.sum(axis=0) == 0
        ):
            continue

        X_observed = X_complete[
            observed_mask,
            :,
        ]

        if np.linalg.matrix_rank(
            X_observed
        ) != p_fixed:
            continue

        missing_grid = ~observed_grid

        return {
            "observed_mask": observed_mask,
            "observed_pplus_grid": observed_grid,
            "missing_pplus_grid": missing_grid,
            "n_missing_pplus": int(
                missing_grid.sum()
            ),
            "n_observed_pplus": int(
                observed_grid.sum()
            ),
        }

    raise RuntimeError(
        "Could not generate a valid Simulation 2 missingness mask "
        "after 500 attempts."
    )


def prepare_incomplete_dataset(seed):
    dataset = simulate_complete_dataset(seed)

    missingness = apply_missing_pathogen_plasmid_combinations(
        dataset,
        mask_seed=seed + 50_000_000,
    )

    observed_mask = missingness[
        "observed_mask"
    ]

    dataset.update(
        missingness
    )

    dataset["X"] = dataset[
        "X_complete"
    ][observed_mask, :]

    dataset["y"] = dataset[
        "y_complete"
    ][observed_mask]

    dataset["pathogen_index"] = dataset[
        "pathogen_index_complete"
    ][observed_mask]

    dataset["plasmid_state_index"] = dataset[
        "plasmid_state_index_complete"
    ][observed_mask]

    return dataset


print("Cell 3: PASS")


In [ ]:
#@title Cell 4 - Define efficient REML and GLS fitting

def prepare_reml_static(X, pathogen_index, K):
    X = np.asarray(X, dtype=float)
    K = np.asarray(K, dtype=float)

    m, p_fixed = X.shape

    design_rank = np.linalg.matrix_rank(X)

    if design_rank != p_fixed:
        raise ValueError(
            f"Fixed-effect design matrix is rank deficient: "
            f"rank={design_rank}, columns={p_fixed}."
        )

    eigenvalues, Q = np.linalg.eigh(
        (K + K.T) / 2.0
    )

    keep = eigenvalues > 1e-10
    eigenvalues = eigenvalues[keep]
    Q = Q[:, keep]

    B_lowrank = (
        Q[pathogen_index, :]
        * np.sqrt(eigenvalues)[None, :]
    )

    static = {
        "m": int(m),
        "p_fixed": int(p_fixed),
        "rank_K": int(len(eigenvalues)),
        "B_lowrank": B_lowrank,
        "BtB": B_lowrank.T @ B_lowrank,
        "BtX": B_lowrank.T @ X,
        "XTX": X.T @ X,
        "X": X,
    }

    return static


def add_y_to_reml_static(static, y):
    working = {
        key: value
        for key, value in static.items()
        if key not in {"B_lowrank", "X"}
    }

    B_lowrank = static["B_lowrank"]
    X = static["X"]

    working["Bty"] = B_lowrank.T @ y
    working["Xty"] = X.T @ y
    working["yty"] = float(y @ y)

    return working


def evaluate_profile_reml(log_delta, working, return_fit=False):
    delta = float(np.exp(log_delta))

    m = working["m"]
    p_fixed = working["p_fixed"]
    rank_K = working["rank_K"]

    M = (
        np.eye(rank_K)
        + working["BtB"] / delta
    )

    try:
        chol_M = cho_factor(
            M,
            lower=True,
            check_finite=False,
        )
    except np.linalg.LinAlgError:
        return np.inf if not return_fit else None

    M_inv_BtX = cho_solve(
        chol_M,
        working["BtX"],
        check_finite=False,
    )

    M_inv_Bty = cho_solve(
        chol_M,
        working["Bty"],
        check_finite=False,
    )

    XtAinvX = (
        working["XTX"] / delta
        - (
            working["BtX"].T
            @ M_inv_BtX
        ) / (delta ** 2)
    )

    XtAinvy = (
        working["Xty"] / delta
        - (
            working["BtX"].T
            @ M_inv_Bty
        ) / (delta ** 2)
    )

    yAinvy = (
        working["yty"] / delta
        - float(
            working["Bty"].T
            @ M_inv_Bty
        ) / (delta ** 2)
    )

    XtAinvX = (
        XtAinvX + XtAinvX.T
    ) / 2.0

    sign_X, logdet_X = np.linalg.slogdet(
        XtAinvX
    )

    if sign_X <= 0:
        return np.inf if not return_fit else None

    try:
        chol_X = cho_factor(
            XtAinvX,
            lower=True,
            check_finite=False,
        )

        beta_hat = cho_solve(
            chol_X,
            XtAinvy,
            check_finite=False,
        )
    except np.linalg.LinAlgError:
        return np.inf if not return_fit else None

    q = float(
        yAinvy
        - beta_hat @ XtAinvy
    )

    df_reml = m - p_fixed

    if q <= 0 or df_reml <= 0:
        return np.inf if not return_fit else None

    logdet_M = 2.0 * np.sum(
        np.log(np.diag(chol_M[0]))
    )

    logdet_A = (
        m * np.log(delta)
        + logdet_M
    )

    objective = (
        logdet_A
        + logdet_X
        + df_reml * np.log(q / df_reml)
    )

    if return_fit:
        return {
            "objective": float(objective),
            "beta_hat": beta_hat,
            "q": q,
            "delta": delta,
            "df_reml": int(df_reml),
        }

    return float(objective)


def fit_section2_reml_gls(X, y, pathogen_index, K, static=None):
    if static is None:
        static = prepare_reml_static(
            X,
            pathogen_index,
            K,
        )

    working = add_y_to_reml_static(
        static,
        y,
    )

    optimization = minimize_scalar(
        lambda log_delta: evaluate_profile_reml(
            log_delta,
            working,
            return_fit=False,
        ),
        bounds=(-8.0, 8.0),
        method="bounded",
        options={
            "xatol": 1e-4,
            "maxiter": 100,
        },
    )

    if not optimization.success:
        raise RuntimeError(
            "REML optimization failed: "
            + str(optimization.message)
        )

    fit = evaluate_profile_reml(
        optimization.x,
        working,
        return_fit=True,
    )

    if fit is None:
        raise RuntimeError(
            "Final REML/GLS evaluation failed."
        )

    sigma_g2_hat = (
        fit["q"]
        / fit["df_reml"]
    )

    sigma_e2_hat = (
        fit["delta"]
        * sigma_g2_hat
    )

    return {
        "beta_hat": fit["beta_hat"],
        "sigma_g2_hat": float(sigma_g2_hat),
        "sigma_e2_hat": float(sigma_e2_hat),
        "delta_hat": float(fit["delta"]),
        "reml_objective": float(fit["objective"]),
        "optimization_nfev": int(optimization.nfev),
        "static": static,
    }


def unpack_beta(beta_hat):
    start_C = 1
    stop_C = start_C + D_C

    start_P = stop_C
    stop_P = start_P + D_P

    start_B = stop_P

    alpha_hat = float(beta_hat[0])
    beta_C_hat = beta_hat[start_C:stop_C]
    beta_P_hat = beta_hat[start_P:stop_P]
    B_hat = beta_hat[start_B:].reshape(
        D_C,
        D_P,
    )

    return (
        alpha_hat,
        beta_C_hat,
        beta_P_hat,
        B_hat,
    )


print("Cell 4: PASS")


In [ ]:
#@title Cell 5 - Generate and QC one incomplete-design dataset

EXAMPLE_SEED = MASTER_SEED

example = prepare_incomplete_dataset(
    EXAMPLE_SEED
)

X = example["X"]
y = example["y"]
K = example["K"]

n_complete = (
    N_PATHOGENS
    * (N_PLASMIDS + 1)
)

n_observed = len(y)
n_missing_total = (
    n_complete - n_observed
)

observed_fraction = (
    n_observed / n_complete
)

pplus_total = (
    N_PATHOGENS
    * N_PLASMIDS
)

realized_missing_pplus_fraction = (
    example["n_missing_pplus"]
    / pplus_total
)

mask_table = pd.DataFrame(
    np.column_stack([
        np.ones(
            N_PATHOGENS,
            dtype=int,
        ),
        example[
            "observed_pplus_grid"
        ].astype(int),
    ]),
    index=[
        f"C{i}"
        for i in range(1, N_PATHOGENS + 1)
    ],
    columns=[
        "P0",
        *[
            f"P{j}"
            for j in range(1, N_PLASMIDS + 1)
        ],
    ],
)

MASK_PATH = (
    OUTPUT_DIR
    / "02_example_observation_mask_200x21.csv"
)

mask_table.to_csv(
    MASK_PATH,
    index=True,
)

print("=" * 90)
print("CELL 5 — SIMULATION 2 INCOMPLETE-DESIGN QC")
print("=" * 90)
print(f"Complete observations before masking: {n_complete:,}")
print(f"Observed observations used for fit:   {n_observed:,}")
print(f"Missing observations:                 {n_missing_total:,}")
print(f"Observed fraction of all states:      {observed_fraction:.3f}")
print(f"Missing P+ combinations:              {example['n_missing_pplus']:,}")
print(f"Observed P+ combinations:             {example['n_observed_pplus']:,}")
print(f"Realized missing P+ fraction:         {realized_missing_pplus_fraction:.3f}")
print(f"P0 observations retained:             {mask_table['P0'].sum():,}/{N_PATHOGENS}")
print(f"Observed design columns:               {X.shape[1]:,}")
print(f"Observed design rank:                  {np.linalg.matrix_rank(X):,}")
print(f"K dimensions:                         {K.shape}")
print(f"Observed log2 MIC mean:                {np.mean(y):.6f}")
print(f"Observed log2 MIC SD:                  {np.std(y, ddof=1):.6f}")

print("\nObserved P+ counts per pathogen:")
print(
    pd.Series(
        example["observed_pplus_grid"].sum(axis=1)
    ).describe().to_string()
)

print("\nObserved pathogen counts per plasmid:")
print(
    pd.Series(
        example["observed_pplus_grid"].sum(axis=0)
    ).describe().to_string()
)

print("\nFirst five rows of the observation mask (1=observed, 0=missing):")
display(
    mask_table.head()
)

print("\nSaved:")
print(MASK_PATH)

print("\nCell 5: PASS")


In [ ]:
#@title Cell 6 - Fit the model using only the observed incomplete dataset

start_time = time.time()

example_static = prepare_reml_static(
    example["X"],
    example["pathogen_index"],
    example["K"],
)

example_fit = fit_section2_reml_gls(
    example["X"],
    example["y"],
    example["pathogen_index"],
    example["K"],
    static=example_static,
)

elapsed = time.time() - start_time

(
    alpha_hat,
    beta_C_hat,
    beta_P_hat,
    B_hat,
) = unpack_beta(
    example_fit["beta_hat"]
)

print("=" * 90)
print("CELL 6 — INCOMPLETE-DESIGN REML/GLS FIT")
print("=" * 90)
print(f"Observed rows fitted:          {len(example['y']):,}")
print(f"True alpha:                    {ALPHA_TRUE:.6f}")
print(f"Estimated alpha:               {alpha_hat:.6f}")
print(f"True sigma_g^2:                {SIGMA_G2_TRUE:.6f}")
print(f"Estimated sigma_g^2:           {example_fit['sigma_g2_hat']:.6f}")
print(f"True sigma_e^2:                {SIGMA_E2_TRUE:.6f}")
print(f"Estimated sigma_e^2:           {example_fit['sigma_e2_hat']:.6f}")
print(f"Estimated variance ratio:      {example_fit['delta_hat']:.6f}")
print(f"REML optimizer evaluations:    {example_fit['optimization_nfev']}")
print(f"Fit time:                      {elapsed:.2f} seconds")

plasmid_coefficient_comparison = pd.DataFrame({
    "feature": PLASMID_FEATURE_LABELS,
    "true_beta_P": BETA_P_TRUE,
    "estimated_beta_P": beta_P_hat,
})

print("\nPlasmid main-effect recovery:")
display(
    plasmid_coefficient_comparison
)

print("\nCell 6: PASS")


In [ ]:
#@title Cell 7 - Evaluate recovery for hidden pathogen–plasmid combinations

def true_and_estimated_effects(dataset, fit):
    C = dataset["C"]
    P = dataset["P"]

    (
        alpha_hat,
        beta_C_hat,
        beta_P_hat,
        B_hat,
    ) = unpack_beta(
        fit["beta_hat"]
    )

    y0_true = (
        ALPHA_TRUE
        + C @ dataset["beta_C_true"]
    )

    delta_true = (
        P @ dataset["beta_P_true"]
    )[None, :] + (
        C
        @ dataset["B_true"]
        @ P.T
    )

    yij_true = (
        y0_true[:, None]
        + delta_true
    )

    y0_hat = (
        alpha_hat
        + C @ beta_C_hat
    )

    delta_hat = (
        P @ beta_P_hat
    )[None, :] + (
        C
        @ B_hat
        @ P.T
    )

    yij_hat = (
        y0_hat[:, None]
        + delta_hat
    )

    return {
        "y0_true": y0_true,
        "y0_hat": y0_hat,
        "yij_true": yij_true,
        "yij_hat": yij_hat,
        "delta_true": delta_true,
        "delta_hat": delta_hat,
    }


def basic_metrics(true_values, estimated_values):
    true_values = np.asarray(
        true_values,
        dtype=float,
    ).ravel()

    estimated_values = np.asarray(
        estimated_values,
        dtype=float,
    ).ravel()

    error = (
        estimated_values
        - true_values
    )

    nonzero = (
        np.abs(true_values)
        > 1e-12
    )

    if nonzero.any():
        sign_accuracy = np.mean(
            np.sign(
                estimated_values[nonzero]
            )
            == np.sign(
                true_values[nonzero]
            )
        )
    else:
        sign_accuracy = np.nan

    return {
        "bias": float(
            np.mean(error)
        ),
        "rmse": float(
            np.sqrt(
                np.mean(error ** 2)
            )
        ),
        "sign_accuracy": float(
            sign_accuracy
        ),
    }


def pairwise_delta_delta(delta_matrix):
    upper_i, upper_k = np.triu_indices(
        delta_matrix.shape[0],
        k=1,
    )

    dd = (
        delta_matrix[upper_i, :]
        - delta_matrix[upper_k, :]
    )

    return (
        dd,
        upper_i,
        upper_k,
    )


def evaluate_dataset_fit(dataset, fit):
    effects = true_and_estimated_effects(
        dataset,
        fit,
    )

    (
        dd_true,
        upper_i,
        upper_k,
    ) = pairwise_delta_delta(
        effects["delta_true"]
    )

    dd_hat, _, _ = (
        pairwise_delta_delta(
            effects["delta_hat"]
        )
    )

    missing_delta_mask = dataset[
        "missing_pplus_grid"
    ]

    observed_delta_mask = dataset[
        "observed_pplus_grid"
    ]

    # A DeltaDelta target is treated as a missing-combination target when
    # at least one of its two pathogen-plasmid states was hidden.
    dd_missing_mask = (
        missing_delta_mask[
            upper_i,
            :,
        ]
        | missing_delta_mask[
            upper_k,
            :,
        ]
    )

    dd_both_observed_mask = (
        observed_delta_mask[
            upper_i,
            :,
        ]
        & observed_delta_mask[
            upper_k,
            :,
        ]
    )

    metrics_delta_all = basic_metrics(
        effects["delta_true"],
        effects["delta_hat"],
    )

    metrics_delta_missing = basic_metrics(
        effects["delta_true"][
            missing_delta_mask
        ],
        effects["delta_hat"][
            missing_delta_mask
        ],
    )

    metrics_delta_observed = basic_metrics(
        effects["delta_true"][
            observed_delta_mask
        ],
        effects["delta_hat"][
            observed_delta_mask
        ],
    )

    metrics_dd_all = basic_metrics(
        dd_true,
        dd_hat,
    )

    metrics_dd_missing = basic_metrics(
        dd_true[
            dd_missing_mask
        ],
        dd_hat[
            dd_missing_mask
        ],
    )

    metrics_dd_observed = basic_metrics(
        dd_true[
            dd_both_observed_mask
        ],
        dd_hat[
            dd_both_observed_mask
        ],
    )

    row = {
        "delta_all_bias": metrics_delta_all["bias"],
        "delta_all_rmse": metrics_delta_all["rmse"],
        "delta_all_sign_accuracy": metrics_delta_all["sign_accuracy"],

        "delta_missing_bias": metrics_delta_missing["bias"],
        "delta_missing_rmse": metrics_delta_missing["rmse"],
        "delta_missing_sign_accuracy": metrics_delta_missing["sign_accuracy"],

        "delta_observed_bias": metrics_delta_observed["bias"],
        "delta_observed_rmse": metrics_delta_observed["rmse"],
        "delta_observed_sign_accuracy": metrics_delta_observed["sign_accuracy"],

        "delta_delta_all_bias": metrics_dd_all["bias"],
        "delta_delta_all_rmse": metrics_dd_all["rmse"],
        "delta_delta_all_sign_accuracy": metrics_dd_all["sign_accuracy"],

        "delta_delta_missing_bias": metrics_dd_missing["bias"],
        "delta_delta_missing_rmse": metrics_dd_missing["rmse"],
        "delta_delta_missing_sign_accuracy": metrics_dd_missing["sign_accuracy"],

        "delta_delta_observed_bias": metrics_dd_observed["bias"],
        "delta_delta_observed_rmse": metrics_dd_observed["rmse"],
        "delta_delta_observed_sign_accuracy": metrics_dd_observed["sign_accuracy"],

        "n_missing_delta_targets": int(
            missing_delta_mask.sum()
        ),

        "n_missing_delta_delta_targets": int(
            dd_missing_mask.sum()
        ),

        "sigma_g2_hat": fit["sigma_g2_hat"],
        "sigma_e2_hat": fit["sigma_e2_hat"],
    }

    return {
        "metrics": row,
        "effects": effects,
        "dd_true": dd_true,
        "dd_hat": dd_hat,
        "pair_i": upper_i,
        "pair_k": upper_k,
        "dd_missing_mask": dd_missing_mask,
        "dd_both_observed_mask": dd_both_observed_mask,
    }


example_evaluation = evaluate_dataset_fit(
    example,
    example_fit,
)

example_metrics = example_evaluation[
    "metrics"
]

print("=" * 90)
print("CELL 7 — RECOVERY OF HIDDEN COMBINATIONS")
print("=" * 90)

display(
    pd.DataFrame(
        [example_metrics]
    ).T.rename(
        columns={0: "value"}
    )
)

print("\nPrimary Simulation 2 check:")
print(
    f"Hidden Delta RMSE:      "
    f"{example_metrics['delta_missing_rmse']:.6f}"
)
print(
    f"Hidden DeltaDelta RMSE: "
    f"{example_metrics['delta_delta_missing_rmse']:.6f}"
)

print("\nCell 7: PASS")


In [ ]:
#@title Cell 8 - Run 100 independent Simulation 2 replicates

def run_one_simulation2_replicate(replicate_index):
    seed = (
        MASTER_SEED
        + 1000
        + int(replicate_index)
    )

    dataset = prepare_incomplete_dataset(
        seed
    )

    static = prepare_reml_static(
        dataset["X"],
        dataset["pathogen_index"],
        dataset["K"],
    )

    fit = fit_section2_reml_gls(
        dataset["X"],
        dataset["y"],
        dataset["pathogen_index"],
        dataset["K"],
        static=static,
    )

    evaluation = evaluate_dataset_fit(
        dataset,
        fit,
    )

    metrics = evaluation[
        "metrics"
    ].copy()

    metrics["replicate"] = int(
        replicate_index + 1
    )

    metrics["seed"] = int(seed)

    metrics["realized_missing_pplus_fraction"] = float(
        dataset["n_missing_pplus"]
        / (N_PATHOGENS * N_PLASMIDS)
    )

    return metrics


replicate_rows = []

start_time = time.time()

for r in range(
    N_SIM_REPLICATES
):
    replicate_rows.append(
        run_one_simulation2_replicate(r)
    )

    if (
        (r + 1) % 10 == 0
        or r == 0
        or r + 1 == N_SIM_REPLICATES
    ):
        elapsed = (
            time.time()
            - start_time
        )

        print(
            f"Completed {r + 1:3d}/{N_SIM_REPLICATES} replicates "
            f"({elapsed:.1f} seconds elapsed)"
        )

replicate_results = pd.DataFrame(
    replicate_rows
)

REPLICATE_RESULTS_PATH = (
    OUTPUT_DIR
    / "02_scenario2_missing_combinations_replicate_metrics.csv"
)

replicate_results.to_csv(
    REPLICATE_RESULTS_PATH,
    index=False,
)

print("\nSaved:")
print(REPLICATE_RESULTS_PATH)
print("\nCell 8: PASS")


In [ ]:
#@title Cell 9 - Summarize the 100-replicate Simulation 2 benchmark

PRIMARY_METRICS = [
    "delta_missing_bias",
    "delta_missing_rmse",
    "delta_missing_sign_accuracy",
    "delta_delta_missing_bias",
    "delta_delta_missing_rmse",
    "delta_delta_missing_sign_accuracy",
]

COMPARISON_METRICS = [
    "delta_observed_bias",
    "delta_observed_rmse",
    "delta_observed_sign_accuracy",
    "delta_delta_observed_bias",
    "delta_delta_observed_rmse",
    "delta_delta_observed_sign_accuracy",
]

SUPPORTING_METRICS = [
    "sigma_g2_hat",
    "sigma_e2_hat",
    "realized_missing_pplus_fraction",
]

summary_rows = []

for metric in (
    PRIMARY_METRICS
    + COMPARISON_METRICS
    + SUPPORTING_METRICS
):
    values = replicate_results[
        metric
    ].to_numpy(
        dtype=float
    )

    summary_rows.append({
        "metric": metric,
        "mean": float(
            np.nanmean(values)
        ),
        "sd": float(
            np.nanstd(
                values,
                ddof=1,
            )
        ),
        "median": float(
            np.nanmedian(values)
        ),
        "q025": float(
            np.nanquantile(
                values,
                0.025,
            )
        ),
        "q975": float(
            np.nanquantile(
                values,
                0.975,
            )
        ),
    })

scenario2_summary = pd.DataFrame(
    summary_rows
)

SUMMARY_PATH = (
    OUTPUT_DIR
    / "02_scenario2_missing_combinations_summary.csv"
)

scenario2_summary.to_csv(
    SUMMARY_PATH,
    index=False,
)

print("=" * 90)
print("SCENARIO 2 — 100-REPLICATE SUMMARY")
print("=" * 90)

print("\nPrimary metrics for deliberately hidden combinations:")
display(
    scenario2_summary[
        scenario2_summary[
            "metric"
        ].isin(
            PRIMARY_METRICS
        )
    ].reset_index(
        drop=True
    )
)

print("\nComparison: combinations that remained observed:")
display(
    scenario2_summary[
        scenario2_summary[
            "metric"
        ].isin(
            COMPARISON_METRICS
        )
    ].reset_index(
        drop=True
    )
)

print("\nSupporting diagnostics:")
display(
    scenario2_summary[
        scenario2_summary[
            "metric"
        ].isin(
            SUPPORTING_METRICS
        )
    ].reset_index(
        drop=True
    )
)

print("\nSaved:")
print(SUMMARY_PATH)
print("\nCell 9: PASS")


In [ ]:
#@title Cell 10 - Parametric bootstrap for hidden-combination effects

def simulate_parametric_bootstrap_y(
    rng,
    X,
    pathogen_index,
    K,
    beta_hat,
    sigma_g2_hat,
    sigma_e2_hat,
):
    u_star = draw_correlated_host_effect(
        rng,
        K,
        sigma_g2_hat,
    )

    epsilon_star = rng.normal(
        0.0,
        np.sqrt(
            sigma_e2_hat
        ),
        size=X.shape[0],
    )

    return (
        X @ beta_hat
        + u_star[pathogen_index]
        + epsilon_star
    )


def effects_from_beta(C, P, beta_hat):
    (
        alpha_hat,
        beta_C_hat,
        beta_P_hat,
        B_hat,
    ) = unpack_beta(
        beta_hat
    )

    y0_hat = (
        alpha_hat
        + C @ beta_C_hat
    )

    delta_hat = (
        P @ beta_P_hat
    )[None, :] + (
        C
        @ B_hat
        @ P.T
    )

    return (
        y0_hat,
        delta_hat,
    )


missing_delta_mask = example[
    "missing_pplus_grid"
]

dd_missing_mask = example_evaluation[
    "dd_missing_mask"
]

delta_true_missing = example_evaluation[
    "effects"
]["delta_true"][
    missing_delta_mask
]

dd_true_missing = example_evaluation[
    "dd_true"
][
    dd_missing_mask
]

bootstrap_delta_missing = np.empty(
    (
        N_BOOTSTRAP,
        len(delta_true_missing),
    ),
    dtype=np.float32,
)

bootstrap_dd_missing = np.empty(
    (
        N_BOOTSTRAP,
        len(dd_true_missing),
    ),
    dtype=np.float32,
)

bootstrap_rng = np.random.default_rng(
    MASTER_SEED + 900000
)

start_time = time.time()

for b in range(
    N_BOOTSTRAP
):
    y_star = simulate_parametric_bootstrap_y(
        bootstrap_rng,
        example["X"],
        example["pathogen_index"],
        example["K"],
        example_fit["beta_hat"],
        example_fit["sigma_g2_hat"],
        example_fit["sigma_e2_hat"],
    )

    fit_star = fit_section2_reml_gls(
        example["X"],
        y_star,
        example["pathogen_index"],
        example["K"],
        static=example_static,
    )

    _, delta_star = effects_from_beta(
        example["C"],
        example["P"],
        fit_star["beta_hat"],
    )

    dd_star, _, _ = pairwise_delta_delta(
        delta_star
    )

    bootstrap_delta_missing[
        b,
        :,
    ] = delta_star[
        missing_delta_mask
    ].astype(
        np.float32
    )

    bootstrap_dd_missing[
        b,
        :,
    ] = dd_star[
        dd_missing_mask
    ].astype(
        np.float32
    )

    if (
        (b + 1) % 20 == 0
        or b == 0
        or b + 1 == N_BOOTSTRAP
    ):
        elapsed = (
            time.time()
            - start_time
        )

        print(
            f"Completed {b + 1:3d}/{N_BOOTSTRAP} bootstrap refits "
            f"({elapsed:.1f} seconds elapsed)"
        )

delta_low = np.quantile(
    bootstrap_delta_missing,
    0.025,
    axis=0,
)

delta_high = np.quantile(
    bootstrap_delta_missing,
    0.975,
    axis=0,
)

dd_low = np.quantile(
    bootstrap_dd_missing,
    0.025,
    axis=0,
)

dd_high = np.quantile(
    bootstrap_dd_missing,
    0.975,
    axis=0,
)

delta_contains_truth = (
    (delta_low <= delta_true_missing)
    & (delta_true_missing <= delta_high)
)

dd_contains_truth = (
    (dd_low <= dd_true_missing)
    & (dd_true_missing <= dd_high)
)

delta_excludes_zero = (
    (delta_low > 0)
    | (delta_high < 0)
)

dd_excludes_zero = (
    (dd_low > 0)
    | (dd_high < 0)
)

bootstrap_summary = pd.DataFrame([
    {
        "effect": "Delta_ij_hidden",
        "number_of_effects": int(
            len(delta_true_missing)
        ),
        "fraction_CI_excludes_zero": float(
            np.mean(delta_excludes_zero)
        ),
        "fraction_CI_contains_known_truth": float(
            np.mean(delta_contains_truth)
        ),
    },
    {
        "effect": "DeltaDelta_ikj_at_least_one_hidden",
        "number_of_effects": int(
            len(dd_true_missing)
        ),
        "fraction_CI_excludes_zero": float(
            np.mean(dd_excludes_zero)
        ),
        "fraction_CI_contains_known_truth": float(
            np.mean(dd_contains_truth)
        ),
    },
])

BOOTSTRAP_SUMMARY_PATH = (
    OUTPUT_DIR
    / "02_scenario2_hidden_combination_bootstrap_summary.csv"
)

bootstrap_summary.to_csv(
    BOOTSTRAP_SUMMARY_PATH,
    index=False,
)

display(
    bootstrap_summary
)

print("\nSaved:")
print(BOOTSTRAP_SUMMARY_PATH)
print("\nCell 10: PASS")


In [ ]:
#@title Cell 11 - Optional repeated-dataset bootstrap coverage

print(
    "Formal repeated-dataset bootstrap coverage remains optional and is "
    "disabled by default, as in Simulation 1."
)

if RUN_FULL_BOOTSTRAP_COVERAGE:
    print(
        "RUN_FULL_BOOTSTRAP_COVERAGE=True was requested, but this notebook "
        "does not automatically launch the very expensive 100 x 200 coverage "
        "calculation. The representative hidden-combination bootstrap in "
        "Cell 10 should be inspected first."
    )
else:
    print(
        "RUN_FULL_BOOTSTRAP_COVERAGE=False: no formal repeated-dataset "
        "coverage run was performed."
    )

print("\nCell 11: PASS")


In [ ]:
#@title Cell 12 - Final QC and output manifest

required_output_files = [
    MASK_PATH,
    REPLICATE_RESULTS_PATH,
    SUMMARY_PATH,
    BOOTSTRAP_SUMMARY_PATH,
]

missing_outputs = [
    str(path)
    for path in required_output_files
    if not path.exists()
]

if missing_outputs:
    raise FileNotFoundError(
        "Required Simulation 2 output(s) are missing:\n"
        + "\n".join(
            missing_outputs
        )
    )

manifest = {
    "notebook":
        "08_Simulation_02_Missing_Pathogen_Plasmid_Combinations.ipynb",

    "scenario":
        "Simulation 02 - missing pathogen-plasmid combinations",

    "central_question":
        "Can the framework recover chromosome-dependent plasmid effects "
        "when some pathogen-plasmid combinations are unobserved?",

    "simulation1_biology_unchanged":
        True,

    "simulation2_change": {
        "missingness_target":
            "P1-P20 pathogen-plasmid observations only",

        "missing_fraction":
            MISSING_PPLUS_FRACTION,

        "missingness_mechanism":
            "random removal",

        "P0_retained_for_all_pathogens":
            True,

        "full_rank_observed_design_required":
            True,
    },

    "evaluation_targets": [
        "Delta_ij for deliberately hidden pathogen-plasmid combinations",
        "DeltaDelta_ikj where at least one pathogen-plasmid combination is hidden",
    ],

    "simulation_settings": {
        "pathogens":
            N_PATHOGENS,

        "plasmids":
            N_PLASMIDS,

        "complete_observations_before_masking":
            N_PATHOGENS
            * (N_PLASMIDS + 1),

        "alpha":
            ALPHA_TRUE,

        "sigma_g":
            SIGMA_G_TRUE,

        "sigma_e":
            SIGMA_E_TRUE,

        "simulation_replicates":
            N_SIM_REPLICATES,

        "bootstrap_replicates":
            N_BOOTSTRAP,
    },

    "outputs": [
        str(path)
        for path in required_output_files
    ],
}

MANIFEST_PATH = (
    OUTPUT_DIR
    / "02_scenario2_manifest.json"
)

with open(
    MANIFEST_PATH,
    "w",
) as handle:
    json.dump(
        manifest,
        handle,
        indent=2,
    )

print("=" * 90)
print("SIMULATION 2 NOTEBOOK COMPLETE")
print("=" * 90)
print(f"Manifest: {MANIFEST_PATH}")
print(f"Required output files checked: {len(required_output_files)}")
print(
    "\nOnly the pathogen-plasmid observation pattern differs from Simulation 1."
)
print("Cell 12: PASS")
